# Test toàn luồng YOLO26-OBB + OCR

Detector **nano** có một bảng: `Rank | Model | ACC | CER | Speed | FPS | Speedup`. Mỗi bảng so sánh VietLPR V1/V2, 7 Paddle và 5 model đối chứng. VietLPR_v1 là OCR chính. Cùng detector, crop và ảnh test cho mọi OCR; không dùng box GT để crop. Bảng 03 vẫn chỉ đo từ ảnh biển có sẵn.

## 1. Môi trường và model

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/detection/yolo26_obb.json").is_file())
os.chdir(ROOT)
if Path(sys.executable).resolve() != (ROOT / ".venv-ocr/Scripts/python.exe").resolve():
    raise RuntimeError("Chọn kernel .venv-ocr rồi Restart Kernel")
from ocr.notebook import run
from plate_detection.data import load_config
VARIANTS = list(load_config()["variants"])
print("YOLO26-OBB:", VARIANTS)
from lpr.pipeline import ocr_names
MODELS = ocr_names(include_public=True)
print("OCR:", MODELS)
print("Tổng tổ hợp:", len(VARIANTS) * len(MODELS))


## 2. Nhãn nội dung biển trên ảnh toàn cảnh

Roboflow chỉ có lớp/polygon, không có chuỗi biển để tính ACC/CER. Tạo template một lần rồi điền `text`, `identity_group` (cùng biển cùng nhóm), `reviewed: true` cho val/test. Không dùng OCR dự đoán làm GT. Ảnh nhiều biển cần ghi đủ; ảnh âm có `plates: []`. Cần kiểm nhóm này với train và nguồn train OCR trước khi công bố benchmark.

In [ ]:
from lpr.evaluate import MANIFEST, annotation_template
CREATE_TEMPLATE = False
if CREATE_TEMPLATE:
    print(annotation_template(load_config()))
print("Nhãn toàn luồng:", MANIFEST)


## 3. Kiểm pipeline chính trên val

Dùng detector nano để xem khung, crop và chuỗi đọc được; không dò ngưỡng trên test.

In [ ]:
SHOW_PREVIEW = False
if SHOW_PREVIEW:
    from lpr.pipeline import make_pipeline
    from plate_detection.data import read_image
    from plate_detection.preview import show_prediction
    cfg = load_config(variant="n")
    pipe, _ = make_pipeline("VietLPR_v1", cfg)
    examples = sorted((Path(cfg["dataset"]) / "val/images").glob("*"))[:4]
    for path in examples:
        show_prediction(read_image(path), pipe)


## 4. Chốt cấu hình trên val

Cấu hình detector và OCR cố định; chỉ kiểm val và lưu checksum trước test. Không train lại OCR. Chạy 14 tổ hợp có thể lâu, nhất là Paddle server.

In [ ]:
RUN_VAL = False
if RUN_VAL:
    for variant in VARIANTS:
        run(sys.executable, "-m", "lpr.evaluate", "freeze", "--variant", variant, "--model", *MODELS)


## 5. Đo test CPU

FP32, 4 luồng, warm-up 20, 5 lượt trên mọi ảnh toàn cảnh. Speed gồm YOLO + crop + OCR và tính **ms/ảnh toàn cảnh**, loại đọc ổ đĩa/tải model. Bỏ sót biển tính lỗi; hộp dư được báo FP và ký tự đọc dư tính lỗi chèn. Tiến độ lưu sau từng ảnh, chạy lại tiếp tục.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    for variant in VARIANTS:
        run(sys.executable, "-m", "lpr.evaluate", "test", "--variant", variant, "--model", *MODELS)


## 6. Bảng so sánh

Rank theo ACC giảm → CER tăng → Speed tăng. Trong mỗi bảng, bản chậm nhất có Speedup 1×. Chưa có số đo thì báo thiếu, không tạo số liệu giả.

In [ ]:
from IPython.display import Markdown, display
from lpr.evaluate import output_dir, table
for variant in VARIANTS:
    cfg = load_config(variant=variant)
    display(Markdown(f"### YOLO26{variant}-OBB + OCR"))
    missing = [name for name in MODELS if not (output_dir(cfg) / name / "test_result.json").is_file()]
    if missing:
        print(f"Chưa đủ kết quả: {len(missing)}/{len(MODELS)} OCR. Chạy bước 4–5 sau khi có GT.")
    else:
        display(Markdown(table(MODELS, cfg)))
